# ScholarRAG: M4, answers with citations and refusal

**Goal:** for each question, retrieve 5 passages and let an LLM either answer *only* from them, citing the passages it used, or refuse when they do not contain the answer.

| Step | Setting |
|---|---|
| Retrieval | BM25 top 10 ∪ dense top 10 → `bge-reranker-base` → top 5 chunks (350 words), the final M3 setup |
| LLM | Gemini `gemini-3.5-flash-lite`, temperature 0, JSON output enforced by a schema |
| Output | `answerable` (bool), `answer` (text), `citations` (passage number + exact quote) |
| Checked in code | every citation points to one of the 5 passages, and its quote appears verbatim in that passage |
| Questions | Set A (64 answerable), set B1 (15 answerable), set B2 (20 unanswerable near-miss: the paper is in the corpus, the detail is not) |

This notebook builds and runs the pipeline and reports the automatic checks. Judging whether each answer is *correct* against the gold answer belongs to M5.

**Re-running is safe:** retrieved contexts and LLM answers are cached in `eval/m4_contexts.jsonl` and `eval/m4_answers.jsonl`; a question is sent to the LLM again only if it has no cached answer for the current `PROMPT_VERSION`. Set `FORCE_RERUN = True` to call the LLM for everything.

## 0. Setup

In [1]:
import os, re, json, time, unicodedata
import numpy as np, pandas as pd
from pathlib import Path
from dotenv import load_dotenv

BASE = Path.cwd() / "data"   # jalankan dari root proyek
load_dotenv(BASE.parent / ".env")

MODEL = "gemini-3.5-flash-lite"
PROMPT_VERSION = "v1"
TOP_K = 5             # passage yang diberikan ke LLM
POOL_K = 10           # top-k BM25 dan dense yang digabung lalu di-rerank
SLEEP = 4.5           # jeda antar panggilan: batas free tier 15 request/menit
FORCE_RERUN = False
RUN_IDS = None        # None = semua soal; atau list id untuk uji coba

ch = pd.read_json(BASE / "chunks_clean.jsonl", lines=True, dtype={"paper_id": str})

qa = pd.read_csv(BASE / "eval" / "questions_v1.csv", dtype={"paper_id": str})
qa = qa[["id", "question", "answer", "evidence", "paper_id"]].assign(source="llm", answerable=True)
hw = pd.read_csv(BASE / "eval" / "questions_handwritten_v1.csv", dtype=str).fillna("")
hw["answerable"] = hw.answerable.str.upper().eq("TRUE")
hw["source"] = np.where(hw.answerable, "handwritten", "unanswerable")
qs = pd.concat([qa, hw[["id", "question", "answer", "evidence", "paper_id", "source", "answerable"]]],
               ignore_index=True).rename(columns={"answer": "gold_answer"})
print(qs.source.value_counts().to_dict(), "|", len(ch), "chunk")

{'llm': 64, 'unanswerable': 20, 'handwritten': 15} | 3910 chunk


## 1. Gold chunks

For answerable questions, the chunks that contain the evidence quote (same rule as the earlier notebooks).

In [2]:
def norm(s):
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"-\s+(\w)", r"\1", s)
    return re.sub(r"\s+", " ", s).strip().lower()

ch["norm"] = ch.text.map(norm)

def gold_for(row):
    if not row.answerable:
        return set()
    g = set()
    pids = [p.strip() for p in row.paper_id.split(";") if p.strip()]
    evs = [e.strip() for e in row.evidence.split(";") if e.strip()] if row.source == "handwritten" else [row.evidence]
    for pid, ev in zip(pids, evs):
        g |= set(ch.index[(ch.paper_id == pid) & ch.norm.str.contains(norm(ev), regex=False)])
    return g

gold = {r.id: gold_for(r) for r in qs.itertuples()}
print("soal terjawab tanpa gold:", [i for i in qs[qs.answerable].id if not gold[i]] or "tidak ada")

soal terjawab tanpa gold: tidak ada


## 2. Retrieval: top 5 passages per question

BM25 top 10 ∪ dense top 10, reranked by `bge-reranker-base`. The result is cached, so the reranker (about 3 s per question on an M1) runs only once.

In [3]:
ctx_path = BASE / "eval" / "m4_contexts.jsonl"
contexts = {}
if ctx_path.exists():
    for line in open(ctx_path):
        r = json.loads(line); contexts[r["id"]] = r
chunk_pos = {c: i for i, c in enumerate(ch.chunk_id)}
todo = [q for q in qs.itertuples() if q.id not in contexts
        or any(c not in chunk_pos for c in contexts[q.id]["chunk_ids"])]

if todo:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    from rank_bm25 import BM25Okapi
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

    def tok(s): return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in ENGLISH_STOP_WORDS]

    emb = SentenceTransformer("BAAI/bge-small-en-v1.5")
    assert json.load(open(BASE / "index" / "chunk_ids.json")) == ch.chunk_id.tolist(), "cache embedding tidak cocok"
    E = np.load(BASE / "index" / "emb_bge_small.npy")
    bm25 = BM25Okapi([tok(t) for t in ch.text])
    ce = CrossEncoder("BAAI/bge-reranker-base", max_length=512)
    QP = "Represent this sentence for searching relevant passages: "

    with open(ctx_path, "a") as f:
        for q in todo:
            qv = emb.encode([QP + q.question], normalize_embeddings=True)[0]
            dense = np.argsort(-(E @ qv))[:POOL_K].tolist()
            sparse = np.argsort(-bm25.get_scores(tok(q.question)))[:POOL_K].tolist()
            pool = list(dict.fromkeys(sparse + dense))
            scores = ce.predict([(q.question, ch.text.iat[i]) for i in pool], batch_size=16, show_progress_bar=False)
            order = np.argsort(-scores)[:TOP_K]
            rec = {"id": q.id, "chunk_ids": [ch.chunk_id.iat[pool[j]] for j in order],
                   "rerank_scores": [round(float(scores[j]), 4) for j in order]}
            contexts[q.id] = rec
            f.write(json.dumps(rec) + "\n")
    del ce, emb

ctx_idx = {i: [chunk_pos[c] for c in r["chunk_ids"]] for i, r in contexts.items()}
print(len(todo), "soal di-retrieve sekarang |", len(contexts), "konteks tersimpan")

0 soal di-retrieve sekarang | 99 konteks tersimpan


In [4]:
# apakah chunk gold (atau, untuk soal tak-terjawab, paper-nya) ada di 5 passage?
qs["gold_in_ctx"] = [bool(gold[q.id] & set(ctx_idx[q.id])) for q in qs.itertuples()]
qs["paper_in_ctx"] = [q.paper_id.split(";")[0] in set(ch.paper_id.values[ctx_idx[q.id]]) for q in qs.itertuples()]
qs.groupby("source")[["gold_in_ctx", "paper_in_ctx"]].mean().round(3)

,gold_in_ctx,paper_in_ctx
source,,
handwritten,0.933,1.0
llm,0.906,1.0
unanswerable,0.000,0.9


## 3. Prompt and output schema

Passages are numbered 1–5 in the prompt; the model cites by number and code maps the number back to the chunk id. Each citation carries an exact quote so it can be verified. The refusal rule says explicitly that a passage from the right paper that lacks the specific detail does not count: that is exactly the trap set B2 sets.

In [5]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

class Citation(BaseModel):
    passage: int = Field(description="Number of the passage (1-5) the quote comes from")
    quote: str = Field(description="Exact words copied from that passage, 5-40 words, supporting the answer")

class Answer(BaseModel):
    answerable: bool = Field(description="True only if the passages contain the information asked for")
    answer: str = Field(description="1-3 sentence answer based only on the passages; empty if not answerable")
    citations: list[Citation] = Field(description="Every passage used for the answer; empty if not answerable")

SYSTEM = """You answer questions about research papers on visual anomaly detection, using ONLY the numbered passages you are given.

- If the passages contain the information the question asks for, set answerable to true, answer in 1-3 sentences, and cite every passage you used. Each citation gives the passage number and a quote copied word for word from that passage (5-40 words) that supports the answer.
- If the passages do not contain the specific information asked for, set answerable to false, and leave answer and citations empty.
- A passage about the same paper or topic that does not state the specific detail asked for does NOT make the question answerable. Do not guess, infer missing numbers, or use outside knowledge."""

def build_prompt(question, idxs):
    parts = []
    for n, i in enumerate(idxs, start=1):
        r = ch.iloc[i]
        parts.append(f"[{n}] {r.title} (pages {r.page}-{r.page_end})\n{r.text}")
    return f"Question: {question}\n\nPassages:\n\n" + "\n\n".join(parts)

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
config = types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0,
                                     response_mime_type="application/json", response_schema=Answer)

print(build_prompt(qs.question[0], ctx_idx[qs.id[0]])[:600], "...")

Question: What is the name of the open-source deep learning library introduced for unsupervised anomaly detection and localization that supports OpenVINO model-optimization?

Passages:

[1] Anomalib: A Deep Learning Library for Anomaly Detection (pages 1-1)
ANOMALIB: A DEEP LEARNING LIBRARY FOR ANOMALY DETECTION Samet Akcay, Dick Ameln, Ashwin Vaidya, Barath Lakshmanan, Nilesh Ahuja, Utku Genc This paper introduces anomalib1, a novel library for unsupervised anomaly detection and localization. With reproducibility and modularity in mind, this open-source library provides algorithms from the li ...


## 4. Run the LLM

Answers are appended to `eval/m4_answers.jsonl` one by one, so an interrupted run continues where it stopped. Overload and rate-limit errors (429/503) are retried with a growing pause.

In [6]:
ans_path = BASE / "eval" / "m4_answers.jsonl"
answers = {}
if ans_path.exists() and not FORCE_RERUN:
    for line in open(ans_path):
        r = json.loads(line)
        if r["prompt_version"] == PROMPT_VERSION and r["model"] == MODEL and r.get("parsed") is not None:
            answers[r["id"]] = r

def ask(q):
    prompt = build_prompt(q.question, ctx_idx[q.id])
    for attempt in range(5):
        try:
            t0 = time.perf_counter()
            r = client.models.generate_content(model=MODEL, contents=prompt, config=config)
            lat = time.perf_counter() - t0
            parsed = r.parsed.model_dump() if r.parsed is not None else None
            u = r.usage_metadata
            return {"id": q.id, "model": MODEL, "prompt_version": PROMPT_VERSION, "parsed": parsed, "raw": r.text,
                    "latency_s": round(lat, 2), "tokens_in": u.prompt_token_count, "tokens_out": u.candidates_token_count,
                    "context_chunk_ids": contexts[q.id]["chunk_ids"], "error": None}
        except Exception as e:
            msg = str(e)
            if any(s in msg for s in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")) and attempt < 4:
                time.sleep(20 * (attempt + 1)); continue
            return {"id": q.id, "model": MODEL, "prompt_version": PROMPT_VERSION, "parsed": None, "raw": None,
                    "error": msg[:300], "context_chunk_ids": contexts[q.id]["chunk_ids"]}

run = qs if RUN_IDS is None else qs[qs.id.isin(RUN_IDS)]
todo = [q for q in run.itertuples() if q.id not in answers]
errors = []
with open(ans_path, "a") as f:
    for k, q in enumerate(todo, start=1):
        rec = ask(q)
        f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
        if rec["parsed"] is not None:
            answers[q.id] = rec
        else:
            errors.append((q.id, rec["error"]))
        if k % 10 == 0:
            print(f"{k}/{len(todo)}", flush=True)
        time.sleep(SLEEP)
print(len(todo), "dipanggil sekarang |", len(answers), "jawaban tersimpan | gagal:", errors or "tidak ada")

10/92


20/92


30/92


40/92


50/92


60/92


70/92


80/92


90/92


92 dipanggil sekarang | 99 jawaban tersimpan | gagal: tidak ada


## 5. Automatic checks

Per question:
- **answered**: `answerable` is true.
- **citations valid**: every citation number is 1–5 and its quote appears verbatim (after the same normalisation as the evidence quotes) in that passage. An answer with no citation is not valid.
- **cites gold**: at least one cited passage is a gold chunk (answerable sets only).

In [7]:
def check(q):
    a = answers.get(q.id)
    if a is None:
        return {"id": q.id, "has_output": False}
    p, idxs = a["parsed"], ctx_idx[q.id]
    cits = p["citations"] if p["answerable"] else []
    ok = [1 <= c["passage"] <= len(idxs) and norm(c["quote"]) in ch.norm.iat[idxs[c["passage"] - 1]] for c in cits]
    cited = {idxs[c["passage"] - 1] for c in cits if 1 <= c["passage"] <= len(idxs)}
    return {"id": q.id, "has_output": True, "answered": p["answerable"], "n_citations": len(cits),
            "citations_valid": bool(cits) and all(ok), "cites_gold": bool(cited & gold[q.id]),
            "latency_s": a.get("latency_s"), "tokens_in": a.get("tokens_in"), "tokens_out": a.get("tokens_out"),
            "llm_answer": p["answer"]}

res = qs.merge(pd.DataFrame([check(q) for q in qs.itertuples()]), on="id")
res = res[res.has_output].copy()
res.drop(columns=["norm"], errors="ignore").to_csv(BASE / "eval" / "m4_results.csv", index=False)

def summary(d):
    out = {"n": len(d), "gold/paper in context": d.gold_in_ctx.mean() if d.answerable.all() else d.paper_in_ctx.mean(),
           "answered": d.answered.mean(), "refused": 1 - d.answered.mean()}
    ans = d[d.answered]
    out["citations valid (of answered)"] = ans.citations_valid.mean() if len(ans) else np.nan
    out["cites gold (of answered)"] = ans.cites_gold.mean() if len(ans) and d.answerable.all() else np.nan
    out["avg latency s"] = d.latency_s.mean()
    return pd.Series(out)

order = ["llm", "handwritten", "unanswerable"]
pd.DataFrame({s: summary(res[res.source == s]) for s in order}).T.round(3)

,n,gold/paper in context,answered,refused,citations valid (of answered),cites gold (of answered),avg latency s
llm,64.0,0.906,0.984,0.016,0.984,0.857,3.115
handwritten,15.0,0.933,0.933,0.067,1.000,0.786,2.577
unanswerable,20.0,0.900,0.000,1.000,NaN,NaN,1.125


### 5.1 Where answerable questions go wrong

Two kinds of failure on answerable questions:
- **missed answer**: the gold chunk was among the 5 passages, but the model refused;
- **retrieval miss**: the gold chunk was not retrieved, so refusing is the correct behaviour for the model (the retriever failed, not the LLM).

In [8]:
a = res[res.answerable]
pd.crosstab([a.source, a.gold_in_ctx.map({True: "gold in context", False: "gold not in context"})],
            a.answered.map({True: "answered", False: "refused"}))

answered                         answered  refused
source      gold_in_ctx                           
handwritten gold in context            14        0
            gold not in context         0        1
llm         gold in context            58        0
            gold not in context         5        1

### 5.2 Unanswerable questions the model answered

Each of these is a likely hallucination or an answer built from a passage that does not really contain the detail. They are the most important rows to read by hand.

In [9]:
pd.set_option("display.max_colwidth", 200)
res[(~res.answerable) & res.answered][["id", "question", "llm_answer", "citations_valid"]]

,id,question,llm_answer,citations_valid


### 5.3 Sample answers

In [10]:
res[res.answerable & res.answered].groupby("source").head(3)[["id", "question", "gold_answer", "llm_answer", "citations_valid", "cites_gold"]]

,id,question,gold_answer,llm_answer,citations_valid,cites_gold
0,q001,What is the name of the open-source deep learning library introduced for unsupervised anomaly detection and localization that supports OpenVINO model-optimization?,"Anomalib is an open-source deep learning library designed for unsupervised anomaly detection and localization, offering tools for custom algorithm design and OpenVINO optimization.",The open-source deep learning library introduced for unsupervised anomaly detection and localization that supports OpenVINO model-optimization is called Anomalib.,True,True
1,q002,What is the name of the dual-model ensemble framework that uses two heterogeneous student networks—an Encoder-Decoder and an Encoder-Encoder—to handle both industrial inspection and semantic anoma...,The proposed approach relies on a dual-model ensemble leveraging a shared pre-trained DINOv2 encoder alongside an Encoder-Decoder model for industrial inspection and an Encoder-Encoder model for s...,"The dual-model ensemble framework is presented in a paper titled ""Generalist Multi-Class Anomaly Detection via Distillation to Two Heterogeneous Student Networks"", which proposes a dual-model appr...",True,True
2,q003,How does MambaAD compare to DiAD in terms of parameter efficiency and performance according to the evaluation?,MambaAD achieves approximately a 2.0 increase in performance while utilizing only one-fiftieth of the parameters and computational flops required by DiAD.,"MambaAD achieves an improvement of about 2.0 in the mAD metric compared to DiAD, while using only 1/50th of the parameters and FLOPs of DiAD.",True,True
64,h001,"Which method keeps the descriptions of each product category in memory to spot logical defects, as well as image memories at the object and patch level?",The framework is called Three-Memory framework for Unified structural and logical Anomaly Detection (TMUAD).,"The TMUAD (Three-Memory framework for Unified structural and logical Anomaly Detection) method uses a class-level text memory bank to keep descriptions of each product category, alongside object-l...",True,True
65,h002,Which two benchmark datasets are used to validate the effectiveness of the Iterative Refinement Process (IRP)?,The Iterative Refinement Process (IRP) is validated using the Kolektor SDD2 (KSDD2) and MVTec AD benchmark datasets.,The effectiveness of the Iterative Refinement Process (IRP) is validated using the benchmark datasets Kolektor SDD2 (KSDD2) and MVTec-AD.,True,True
66,h003,What limitation in existing anomaly detection approaches does UniADC address by unifying anomaly detection and classification?,"Existing methods treat anomaly detection and classification as separate tasks, thereby neglecting their inherent correlations and limiting information sharing, leading to suboptimal performance.","Existing methods typically treat anomaly detection and classification as separate tasks, which leads them to neglect their inherent correlations and limit information sharing, resulting in subopti...",True,True


## 6. Reading the results

All 99 questions ran without an API error (prompt `v1`, `gemini-3.5-flash-lite`).

| Set | n | Gold (or paper) in the 5 passages | Answered | Refused | Citations valid | Cites a gold chunk |
|---|---|---|---|---|---|---|
| A, LLM | 64 | 0.906 | 63 | 1 | 62 / 63 | 54 / 63 |
| B1, handwritten | 15 | 0.933 | 14 | 1 | 14 / 14 | 11 / 14 |
| B2, unanswerable | 20 | 0.900 (paper) | **0** | **20** | – | – |

**1. Refusal works on the near-miss trap.** All 20 unanswerable questions are refused, although the right paper was among the passages for 18 of them.

**2. No missed answers.** Of the 72 answerable questions whose gold chunk was retrieved, the model answered all 72. The two refusals on answerable questions (one in A, h005 in B1) are both retrieval misses, where refusing is the right behaviour.

**3. Citations are almost always verbatim.** 76 of 77 answers pass the quote check. The one failure (q034) wrote `√` as LaTeX `\sqrt`; the answer itself is correct.

**4. "Cites a gold chunk" undercounts correct answers.** Reading the 12 answered questions that do not cite a gold chunk:
- Most are correct and cite another chunk of the same paper that states the same fact, usually the abstract (q016, q020, q058, h007, h014), or several body chunks (q007, q042, q048, answered although no gold chunk was retrieved).
- q044 is **incomplete**: the passages do not name the 3D scanner (gold: Creaform MetraSCAN 750), and the model lists the paper's other cameras instead of refusing. This is the failure mode to watch: an answer that is grounded but does not answer the question.
- q035, q046 and h009 differ from the gold answer, but each is supported by a quote from the paper: the gold answer was written from a single passage and the paper says something else (or more) elsewhere. h009 is a genuine inconsistency in the paper itself (the abstract says audio, the experiments use time series).

**5. Cost and speed:** about 2,600 input and 120 output tokens per question; 1–3 s per LLM call on the free tier, plus about 3 s of reranking on an M1.

### What this means for M5

- Gold-chunk matching is not enough to score answers: the same fact sits in several chunks. M5 needs an answer-correctness judgment (manual, or an LLM judge checked against a manual sample) that compares the answer with the gold answer **and** with the cited passages.
- A few set A gold answers are too narrow (q035, q046). M5 should allow "correct but different from gold" when the cited quote supports it, and list those questions for review rather than silently counting them wrong.
- Refusal accuracy on B2 is 20 / 20; harder unanswerable questions (for example the 12 LLM-generated ones in `unanswerable_candidates.csv`) would show whether that holds.